In [4]:
import pandas as pd
import numpy as np


In [5]:
dataset_raw = pd.read_csv('Data/Product_Normalization_GRI.csv')
dataset_raw.head()


,Room Description,Guest Room Info
0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room
1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room
2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room
3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room
4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room


In [5]:
dataset_raw.describe()

,Room Description,Guest Room Info
count,35000,35000
unique,31237,35
top,2X POINTS PACKAGE|2 QUEEN BEDS STUDIO NONSMOKI...,Accessible Room
freq,49,1000


In [4]:
#dataset_raw.fillna('Bungalow', inplace=True)

In [24]:
from collections import Counter
import re
import pandas as pd
from tqdm import tqdm

def detect_potential_abbreviations(descriptions):
    """Extract and analyze potential abbreviations from descriptions"""
    # Extract all word-like tokens
    all_tokens = []
    for desc in tqdm(descriptions, desc="Processing descriptions"):
        if pd.isna(desc):
            continue
        # Extract tokens that look like abbreviations
        tokens = re.findall(r'\b[A-Z0-9/]{2,4}\b', str(desc).upper())
        all_tokens.extend(tokens)
    
    # Count frequencies
    token_counts = Counter(all_tokens)
    
    # Filter and sort by frequency
    potential_abbrevs = {
        token: count for token, count in token_counts.items()
        if count > 10  # Appear in at least 10 descriptions
    }
    
    # Sort by frequency
    sorted_abbrevs = dict(sorted(potential_abbrevs.items(), 
                                key=lambda x: x[1], 
                                reverse=True))
    
    return sorted_abbrevs

# Test the function
potential_abbrevs = detect_potential_abbreviations(dataset_raw['Room Description'])

# Display results
print("\nTop 20 potential abbreviations:")
for abbrev, count in list(potential_abbrevs.items())[:20]:
    print(f"{abbrev}: {count} occurrences")

Processing descriptions: 100%|██████████| 35000/35000 [00:00<00:00, 250897.95it/s]


Top 20 potential abbreviations:
ROOM: 20570 occurrences
RATE: 19210 occurrences
KING: 18190 occurrences
BED: 15464 occurrences
WIFI: 11155 occurrences
FREE: 8472 occurrences
AND: 8417 occurrences
VIEW: 7409 occurrences
WITH: 7245 occurrences
BEST: 5223 occurrences
TO: 3986 occurrences
IN: 3763 occurrences
BEDS: 3763 occurrences
SOFA: 3499 occurrences
TV: 3258 occurrences
SQ: 3155 occurrences
CITY: 2912 occurrences
FT: 2851 occurrences
OR: 2702 occurrences
TWIN: 2690 occurrences


In [27]:
potential_abbrevs

{'ROOM': 20570,
 'RATE': 19210,
 'KING': 18190,
 'BED': 15464,
 'WIFI': 11155,
 'FREE': 8472,
 'AND': 8417,
 'VIEW': 7409,
 'WITH': 7245,
 'BEST': 5223,
 'TO': 3986,
 'IN': 3763,
 'BEDS': 3763,
 'SOFA': 3499,
 'TV': 3258,
 'SQ': 3155,
 'CITY': 2912,
 'FT': 2851,
 'OR': 2702,
 'TWIN': 2690,
 'AAA': 2323,
 'FOR': 2240,
 'COMP': 2181,
 'SQM': 2180,
 'AREA': 2073,
 'STAY': 2071,
 'THE': 2060,
 'NON': 2002,
 'SAFE': 1821,
 'FLEX': 1798,
 'HDTV': 1687,
 'SEMI': 1659,
 'ONE': 1655,
 'WI': 1587,
 'TWO': 1563,
 'SEE': 1499,
 'FULL': 1466,
 'OUR': 1460,
 'HOT': 1460,
 'POOL': 1371,
 'SQFT': 1368,
 'CWT': 1309,
 'YOUR': 1238,
 'TYPE': 1237,
 'LOFT': 1223,
 'TUB': 1213,
 'AT': 1170,
 'WILL': 1170,
 'OF': 1166,
 'DESK': 1146,
 'AC': 1144,
 '2X': 1132,
 'YOU': 1129,
 'FI': 1099,
 'CLUB': 1070,
 'BATH': 1068,
 'RM': 1058,
 'W/': 1030,
 'BB': 993,
 'ONLY': 985,
 'WE': 980,
 'MINI': 974,
 'HIGH': 972,
 'PLUS': 954,
 'SIZE': 917,
 'DO': 916,
 'BAR': 902,
 'WHEN': 895,
 'ON': 886,
 'INCL': 861,
 'MEET': 

In [34]:
pd.DataFrame(potential_abbrevs, index = [0]).T
# = detect_potential_abbreviations(dataset_raw['Room Description'])# %%



,0
ROOM,20570
RATE,19210
KING,18190
BED,15464
WIFI,11155
...,...
/29,11
FCA,11
770,11
SODA,11


In [36]:
pd.DataFrame(potential_abbrevs, index = [0]).T.to_excel('potential_abbrevs.xlsx', index=True)

In [17]:
len(potential_abbrevs)
# %%


965

In [20]:
import re

def expand_abbreviations(text):
    """Expand common hotel abbreviations to full form"""
    
    # Common hotel abbreviations
    abbrev_dict = {
        # Room Types
        'RM': 'ROOM',
        'STE': 'SUITE',
        'STD': 'STANDARD',
        'DLX': 'DELUXE',
        'EXEC': 'EXECUTIVE',
        
        # Bed Types
        'KG': 'KING',
        'QN': 'QUEEN',
        'DBL': 'DOUBLE',
        'TWN': 'TWIN',
        
        # Accessibility
        'ADA': 'ACCESSIBLE',
        'ACC': 'ACCESSIBLE',
        'HEAR': 'HEARING',
        'ACCS': 'ACCESSIBLE',
        
        # View Types
        'MTN': 'MOUNTAIN',
        'OCN': 'OCEAN',
        'VW': 'VIEW',
        
        # Amenities
        'NOSMOK': 'NON SMOKING',
        'NS': 'NON SMOKING',
        'SMKG': 'SMOKING',
        'W/': 'WITH',
        'W': 'WITH',
        'W/O': 'WITHOUT',
        'BAL': 'BALCONY',
        'KITCH': 'KITCHEN',
        'KTCH': 'KITCHEN',
        
        # Common Words
        'FLR': 'FLOOR',
        'LVL': 'LEVEL',
        'BLDG': 'BUILDING',
        'BR': 'BEDROOM'

    }
    
    # Sort by length (longest first) to avoid partial replacements
    sorted_abbrev = sorted(abbrev_dict.items(), key=lambda x: len(x[0]), reverse=True)
    
    # Replace abbreviations
    text = text.upper()  # Convert to uppercase for consistency
    for abbrev, full_form in sorted_abbrev:
        # Add word boundaries to avoid partial word matches
        pattern = r'\b' + re.escape(abbrev) + r'\b'
        text = re.sub(pattern, full_form, text)
    
    return text


In [22]:

# Example usage:
sample_texts = [
    "DLX BR KG RM W/ OCN VW",
    "EXEC STE W/ KITCH",
    "ADA QN RM W/ ROLL IN SHWR",
    "STD DBL NS W/ MTN VW"
]

for text in sample_texts:
    expanded = expand_abbreviations(text)
    print(f"Original: {text}")
    print(f"Expanded: {expanded}\n")

Original: DLX BR KG RM W/ OCN VW
Expanded: DELUXE BEDROOM KING ROOM WITH/ OCEAN VIEW

Original: EXEC STE W/ KITCH
Expanded: EXECUTIVE SUITE WITH/ KITCHEN

Original: ADA QN RM W/ ROLL IN SHWR
Expanded: ACCESSIBLE QUEEN ROOM WITH/ ROLL IN SHWR

Original: STD DBL NS W/ MTN VW
Expanded: STANDARD DOUBLE NON SMOKING WITH/ MOUNTAIN VIEW



In [28]:
dataset_raw['Room Description Expanded'] = dataset_raw['Room Description'].apply(expand_abbreviations)
dataset_raw.head()


,Room Description,Guest Room Info,Room Description Expanded
0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...
1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...
2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...
3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...
4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...


In [31]:
#dataset_raw['Room Description'] == dataset_raw['Room Description Expanded']

0        False
1        False
2        False
3        False
4        False
         ...  
34995     True
34996     True
34997     True
34998     True
34999     True
Length: 35000, dtype: bool

In [32]:
dataset_raw.to_csv('Data/Product_Normalization_GRI_Expanded.csv')